Diagrama UML

![Diagrama UML](docs/diagrama_uml.png)

DDL – DATA DEFINITION LANGUAGE

In [5]:
import sqlite3

# Función para inicializar la base de datos y crear las tablas necesarias
def inicializar_db(libros_db = "libros.db"):
    conexion = sqlite3.connect(libros_db) # Conectar a la base de datos SQLite, si el archivo no existe se creará automáticamente
    cursor = conexion.cursor() # Crear las tablas necesarias para almacenar la información de los libros, autores y categorías

    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS categorias (
            id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT UNIQUE NOT NULL
        );
    ''') # Crear la tabla de categorías con un campo de ID autoincremental y un campo de nombre único
    print("Tabla Categorias creada.")

    cursor.executescript('''    
        CREATE TABLE IF NOT EXISTS autores (
            id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre_autor TEXT UNIQUE NOT NULL
        );
        ''') # Crear la tabla de autores con un campo de ID autoincremental y un campo de nombre único
    print("Tabla Autores creada.")
           
    cursor.executescript(''' 
        CREATE TABLE IF NOT EXISTS libros (
            id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            precio REAL,
            rating INTEGER,
            stock INTEGER,
            link TEXT,
            categoria_id INTEGER,
            FOREIGN KEY (categoria_id) REFERENCES categorias (id_categoria)
        );
    ''') # Crear la tabla de libros con un campo de ID autoincremental, campos para el título, precio, rating, stock, link y una clave foránea que referencia a la tabla de categorías
    print("Tabla Libros creada.")

    cursor.executescript('''        
        CREATE TABLE IF NOT EXISTS libro_autor (
            libro_id INTEGER,
            autor_id INTEGER,
            FOREIGN KEY (libro_id) REFERENCES libros (id_libro),
            FOREIGN KEY (autor_id) REFERENCES autores (id_autor),
            PRIMARY KEY (libro_id, autor_id)
        );                                      
    ''') # Crear la tabla de relación entre libros y autores con claves foráneas que referencian a las tablas de libros y autores, y una clave primaria compuesta por libro_id y autor_id
    print("Tabla Libro_Autor creada.")

    conexion.commit() # Guardar los cambios en la base de datos
    return conexion # Devolver la conexión a la base de datos para su uso posterior

conn = inicializar_db() # Llamar a la función para inicializar la base de datos y almacenar la conexión en la variable conn

Tabla Categorias creada.
Tabla Autores creada.
Tabla Libros creada.
Tabla Libro_Autor creada.


WEB SCRAPING

In [2]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, quote_plus
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
from dotenv import load_dotenv

URL_BASE = "https://books.toscrape.com/" # URL base del sitio web que vamos a scrapear
ESTRELLAS = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5} # Diccionario para convertir el texto de rating en número de estrellas

_cache_autores = {} # Diccionario para almacenar los autores de los libros

# Función para crear una sesión de requests con un User-Agent personalizado
def crear_sesion(): 
    session = requests.Session() # Crear una sesión para mantener las cookies y configuraciones entre peticiones
    # Agregamos un User-Agent personalizado para evitar ser bloqueado por el sitio web.
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'})
    return session # Devolver la sesión creada para su uso en las peticiones

# Función para obtener el contenido HTML de una página y parsearlo con BeautifulSoup
def obtener_soup(url, session):
    try: # Intentamos hacer la petición a la URL utilizando la sesión proporcionada
        respuesta = session.get(url) # Hacemos la petición GET a la URL utilizando la sesión para mantener las configuraciones y cookies
        respuesta.raise_for_status() # Verificamos que la respuesta sea exitosa (código 200), si no, se lanzará una excepción
        soup = BeautifulSoup(respuesta.text, 'lxml') # Parseamos el contenido HTML de la respuesta utilizando BeautifulSoup con el parser 'lxml' para facilitar la extracción de datos
        return soup # Devolver el objeto BeautifulSoup con el contenido de la página
    
    except Exception as error: # Captura cualquier error y muestra un mensaje de error.
        print(f"Error en {url}: {error}")
        return None # Devolver None en caso de error para indicar que no se pudo obtener el contenido de la página

# Función para extraer el stock disponible a partir del texto de disponibilidad    
def extraer_stock(disponibilidad_texto):
    match = re.search(r'\((\d+) available\)', disponibilidad_texto) # Expresión regular para buscar el número de unidades disponibles en el texto de disponibilidad.
    if match:
        return int(match.group(1)) # Convertimos el número encontrado a entero y lo devolvemos como stock disponible
    return 0

# Función para extraer los datos de un libro a partir del elemento HTML que lo representa, la categoría a la que pertenece, la URL actual y la sesión de requests
def extraer_datos_libro(libros, categoria_nombre, url_actual, session):
    titulo = libros.find('h3').a['title'] # Extraemos el titulo del libro mediante las etiquetas 'h3 y a' del HTML de la pagina.
    precio_text = libros.find("p", class_="price_color").text # Extraemos el texto del precio del libro mediante la etiqueta 'p' y 'class precio_color' del HTML de la pagina.
    precio_simb = precio_text.replace("£", "").replace('Â', '').strip() # Limpiamos el texto del precio eliminando simbolos o caracteres extraños.
    precio = float(precio_simb) # Convertimos el precio limpio a un número decimal (float).
    rating_objeto = libros.find("p", class_="star-rating").get("class") # Extraemos el rating del libro buscando el elemento 'p' con la clase 'star-rating' del HTML de la pagina.
    rating_text = rating_objeto[1] # Guardamos el segundo elemento de la clase del rating.
    rating = ESTRELLAS.get(rating_text, 0) # Mediante el diccionario convertimos el texto del rating a un número de estrellas.

    enlace_relativo = libros.find('h3').a['href'] # Extraemos el enlace relativo del libro mediante el atributo 'href' dentro del HTML de la pagina.
    link_libro = urljoin(url_actual, enlace_relativo) # Metodo para contruir la url completa del libro. 

    try:
        respuesta_detalle = session.get(link_libro, timeout=10) # Petición Get a la url del libro y un timeout de 10 segundos.

        if respuesta_detalle.status_code == 200: # Verificamos que la respuesta del detalle del libro sea exitosa (código 200).
            soup_detalle = BeautifulSoup(respuesta_detalle.text, 'lxml') # Utilizamos lxml para parsear el HTML del detalle del libro.
            texto_disponibilidad = soup_detalle.find('p', class_="instock availability").text.strip() # Obtenemos el texto de disponibilidad del libro.
            stock = extraer_stock(texto_disponibilidad) # Función para extraer el stock disponible a partir del texto de disponibilidad.

        else:
            stock = 0 # Asignamos 0 al stock si la respuesta del detalle del libro no es exitosa.sss
    
    except Exception as e: # Captura cualquier error y muestra un mensaje de error.
        print(f"Error buscando stock en {titulo}: {e}")
        stock = 0

    return {
        "titulo": titulo,
        "autor": "Pendiente",
        "precio": precio,
        "categoria": categoria_nombre,
        "rating": rating,
        "stock": stock,
        "link": link_libro
    } # Retornamos un diccionario que contiene los datos del libro.

# Función para obtener los enlaces de todas las categorías disponibles en el sitio web, utilizando la sesión de requests proporcionada para hacer las peticiones y parsear el contenido
def obtener_links_categorias(session):
    soup = obtener_soup(URL_BASE, session) # Obtenemos el contenido HTML de la página principal y lo parseamos con BeautifulSoup
    lista_categoria = [] # Creamos una lista vacía para almacenar los datos de las categorías que vamos a extraer

    panel_lateral = soup.find("div", class_="side_categories") # Buscamos el panel lateral de categorías mediante la 'class' dentro del HTML.
    enlaces = panel_lateral.find_all("a") # Buscamos todos los enlaces dentro del panel lateral de categorías.

    for enlace in enlaces[1:]: # Iteramos sobre los enlaces encontrados, excluyendo el primer enlace que corresponde a "Books by Category"
        nombre_cat = enlace.text.strip() # Extraemos el texto del enlace y lo limpiamos de espacios en blanco adicionales
        ruta_relativa = enlace["href"] # Obtenemos la ruta relativa del enlace, que es la parte de la URL que identifica a la categoría específica
        link_completo = URL_BASE + ruta_relativa # Obtenemos el link completo mediante la URL base y el relativo

        lista_categoria.append({
            "nombre": nombre_cat,
            "url": link_completo
        }) # Agregamos el diccionario con el nombre y la URL de la categoría a la lista de categorías.

    return lista_categoria # Retornamos el diccionario.

# Funcion para procesar una categoría específica, extrayendo los datos de los libros de esa categoría y manejando la paginación si es necesario
def procesar_categoria(categoria, session):
    nombre = categoria["nombre"] # Extraemos el nombre de la categoría del diccionario que recibe la función.
    url_actual = categoria["url"] # Extraemos la URL de la categoría del diccionario que recibe la función.
    libros_categoria = [] # Creamos una lista vacía para almacenar los datos de los libros que se extraigan de esta categoría.
    
    while True: # Bucle infinito para manejar la paginación de la categoría
        soup = obtener_soup(url_actual, session) # Obtenemos el contenido de la página actual de la categoría utilizando la función obtener_soup.
        if not soup:
            break # Salimos del bucle si no se pudo obtener el contenido de la página

        articulos = soup.find_all("article", class_="product_pod") # Buscamos los artículos que representan los libros en la página de la categoría
        for articulo in articulos: # Iteramos sobre los artículos encontrados
            datos = extraer_datos_libro(articulo, nombre, url_actual, session) # Llamamos a la función extraer_datos_libro para obtener los datos del libro
            libros_categoria.append(datos) # Agregamos los datos del libro a la lista de libros de la categoría

        boton_next = soup.find("li", class_="next") # Buscamos el botón de "next" que nos indica que hay más páginas de libros en la categoría actual
        if boton_next:
            enlace_next = boton_next.a['href'] # Obtenemos el enlace de la siguiente página
            url_actual = urljoin(url_actual, enlace_next) # Obtenemos la URL completa de la siguiente página mediante la URL base y el enlace relativo
        else:
            break # Si no encontramos el botón de "next", significa que hemos llegado a la última página de la categoría, por lo que salimos del bucle
            
    # Imprimimos el resultado al finalizar la categoría
    print(f"✅ Categoría procesada: {nombre:.<30} {len(libros_categoria)} libros")
    return libros_categoria # Retornamos la lista de libros de la categoría

# Función para ejecutar el scraping del sitio web y obtener los datos de los libros
def ejecutar_scraping():
    # --- FASE 1: SCRAPING DEL SITIO (Descarga masiva) ---
    print("🚀 FASE 1: Descargando libros de todas las categorías...")
    inicio = time.time() # Guardamos el tiempo de inicio para medir el tiempo que tarda esta fase del proceso
    
    mi_sesion = crear_sesion() # Creamos una sesión de requests para hacer las peticiones HTTP
    categorias = obtener_links_categorias(mi_sesion) # Obtenemos los enlaces de todas las categorías disponibles en el sitio web
    
    lista_libros = [] # Lista vacia donde se guardaran los libros.
    
    # Usamos ThreadPoolExecutor para procesar las categorías en paralelo (5 hilos maximo)
    with ThreadPoolExecutor(max_workers=5) as executor:
        trabajos_pendientes = [] # Lista donde se guardaran los trabajos pendientes.
        
        for categoria in categorias: # Iteramos sobre las categorías disponibles.
            trabajo = executor.submit(procesar_categoria, categoria, mi_sesion) # Enviamos cada categoría a la función procesar_categoria para que la procese en paralelo.
            trabajos_pendientes.append(trabajo) # Agregamos el trabajo a la lista de trabajos pendientes.
            
        for trabajo_terminado in as_completed(trabajos_pendientes): # Iteramos sobre los trabajos terminados.
            libros_de_una_categoria = trabajo_terminado.result() # Obtenemos el resultado del trabajo terminado.      
            lista_libros.extend(libros_de_una_categoria) # Agregamos los libros de la categoría procesada a la lista principal de libros.
        
    tiempo_fase1 = time.time() - inicio # Medimos el tiempo que tardó la fase 1.
    
    print(f"\n📦 FASE 1 COMPLETADA: {len(lista_libros)} libros descargados en {tiempo_fase1:.2f}s.")
    
    return lista_libros # Retornamos la lista de libros con toda la información extraída y actualizada

# Punto de entrada del script
if __name__ == "__main__":
    lista_libros = ejecutar_scraping() # Ejecutamos el scraping y obtenemos la lista de libros

🚀 FASE 1: Descargando libros de todas las categorías...
✅ Categoría procesada: Travel........................ 11 libros
✅ Categoría procesada: Philosophy.................... 11 libros
✅ Categoría procesada: Classics...................... 19 libros
✅ Categoría procesada: Historical Fiction............ 26 libros
✅ Categoría procesada: Mystery....................... 32 libros
✅ Categoría procesada: Womens Fiction................ 17 libros
✅ Categoría procesada: Religion...................... 7 libros
✅ Categoría procesada: Romance....................... 35 libros
✅ Categoría procesada: Childrens..................... 29 libros
✅ Categoría procesada: Music......................... 13 libros
✅ Categoría procesada: Sequential Art................ 75 libros
✅ Categoría procesada: Sports and Games.............. 5 libros
✅ Categoría procesada: Science Fiction............... 16 libros
✅ Categoría procesada: Fiction....................... 65 libros
✅ Categoría procesada: New Adult..................

API GOOGLE BOOKS PARA BUSCAR AUTORES

In [3]:
load_dotenv() # Carga las variables de entorno desde el archivo .env
API_KEY = os.getenv("GOOGLE_API_KEY") # Obtener la API KEY de Google Books desde las variables de entorno

if not API_KEY: # Si no se encuentra la API KEY, mostramos una advertencia
    print("⚠️ ADVERTENCIA: No se encontró la API KEY. Configura tu archivo .env")

# Función para obtener el autor de un libro a partir del título del libro y la API KEY de Google Books
def obtener_autor(libro):
    titulo = libro['titulo'] # Obtenemos el título del libro del diccionario pasado como parámetro.
    
    titulo_limpio = re.sub(r'\s*\(.*?\)', '', titulo).strip() # Elimina el texto entre paréntesis.
    titulo_limpio = re.sub(r'\s*#\d+', '', titulo_limpio).strip() # Elimina el número de la lista.
    if ':' in titulo_limpio and len(titulo_limpio) > 60:  # Si el título contiene ':' y tiene más de 60 caracteres, se toma la parte antes de ':'
        titulo_limpio = titulo_limpio.split(':')[0].strip()

    # ── Caché: si ya buscamos este título, no volvemos a llamar a la API ─────
    if titulo_limpio in _cache_autores:
        libro['autor'] = _cache_autores[titulo_limpio] # Asignamos el autor del libro al diccionario.
        return f"💾 Caché: {titulo[:30]}..."


    if API_KEY: # Verificamos que la API KEY de Google Books esté configurada.
        try:
            url = (
                f"https://www.googleapis.com/books/v1/volumes" # URL base de la API de Google Books.
                f"?q=intitle:{quote_plus(titulo_limpio)}" # Parametro para buscar el libro por titulo.
                f"&key={API_KEY}" # Parametro para la API KEY de Google Books.
                f"&maxResults=3"  # traemos 3 por si el primero no tiene autor
                f"&fields=items(volumeInfo/authors)" # Parametro para obtener solo los autores de los libros.
                f"&langRestrict=en"  # el sitio es en inglés, esto mejora los resultados
            )

            resp = requests.get(url, timeout=5) # Petición GET a la URL de la API de Google Books.

            if resp.status_code == 200: # Verificamos que la respuesta sea exitosa (código 200).
                items = resp.json().get("items", []) # Obtenemos los items de la respuesta.
                for item in items:  # revisamos los 3 resultados, no solo el primero
                    autores = item.get("volumeInfo", {}).get("authors", []) # Obtenemos los autores del libro.
                    if autores:
                        autor = autores[0] # Obtenemos el primer autor.
                        libro['autor'] = autor # Asignamos el autor al libro.
                        _cache_autores[titulo_limpio] = autor # Guardamos el autor en el cache.
                        return f"✅ Google: {titulo[:30]}... -> {autor}" # Retornamos el autor del libro.

            elif resp.status_code == 429: # Verificamos si se alcanzó el límite de peticiones (código 429).
                print("⚠️  Rate limit de Google Books alcanzado, esperando 10s...") 
                time.sleep(10) # Esperamos 10 segundos para evitar sobrecargar el servidor.

        except Exception as e: # Captura cualquier error y muestra un mensaje de error.
            print(f"❌ Error en '{titulo[:30]}': {e}")

        time.sleep(1.0)  # pausa respetuosa entre requests
    
    # Fallback: Si no hay key, falló la búsqueda o cayó en el 70% restante
    libro['autor'] = "Autor Simulado"
    _cache_autores[titulo_limpio] = "Autor Simulado"
    return f"🤖 Simulado: {titulo[:15]}..."

def actualizar_autores(lista_libros):
    # --- FASE 2: ACTUALIZACIÓN DE AUTORES (Aquí se quita el "Pendiente") ---
    print("\n🕵️ FASE 2: Buscando autores para cada libro...")
    
    # Usamos ThreadPoolExecutor para actualizar los autores en paralelo (max_workers=2 es seguro para la API)
    with ThreadPoolExecutor(max_workers=2) as executor:
        busqueda_autores = [executor.submit(obtener_autor, libro) for libro in lista_libros] # Enviamos cada libro a la función que busca el autor
        total = len(busqueda_autores) # Obtenemos el número total de libros a procesar
        for i, future in enumerate(as_completed(busqueda_autores), 1): # Iteramos sobre los trabajos terminados
            future.result() # Obtenemos el resultado del trabajo terminado
            if i % 10 == 0: # Imprimimos progreso cada 10 libros
                print(f"   ...Progreso Autores: {i}/{total} completados")

    print(f"\n🎉 ¡Misión Cumplida! Todos los datos están listos.")

if __name__ == "__main__":
    actualizar_autores(lista_libros) # Actualizamos los autores de cada libro utilizando la función actualizar_autores


🕵️ FASE 2: Buscando autores para cada libro...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
   ...Progreso Autores: 10/1000 completados
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
   ...Progreso Autores: 20/1000 completados
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
   ...Progreso Autores: 30/1000 completados
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
   ...Progreso Autores: 40/1000 completados
⚠️  Rate limit de Google Books alcanzado, esperando 10s...
   ...Progreso Autores: 50/1000 completados
⚠️  Rate limit de Google

LISTA DE LIBROS.

In [4]:
print("\nDatos Libros")
for i, libro in enumerate(lista_libros, 1): # Iteramos sobre la lista de libros para mostrar los datos de cada libro
    titulo = libro.get('titulo', 'N/A') # Obtenemos el título del libro
    autor = libro.get('autor', 'N/A') # Obtenemos el autor del libro
    precio = libro.get('precio', 0.0) # Obtenemos el precio del libro
    categoria = libro.get('categoria', 'N/A') # Obtenemos la categoría del libro
    rating = libro.get('rating', 0) # Obtenemos la calificación del libro
    stock = libro.get('stock', 'N/A') # Obtenemos el stock del libro
    
    print(f"{i}- | {titulo} | {autor} | £{precio} | {categoria} | ⭐ {rating} | {stock}")


Datos Libros
1- | It's Only the Himalayas | S. Bedford | £45.17 | Travel | ⭐ 2 | 19
2- | Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond | Rick Antonson | £49.43 | Travel | ⭐ 4 | 15
3- | See America: A Celebration of Our National Parks & Treasured Sites | Orville O. Hiestand | £48.87 | Travel | ⭐ 3 | 14
4- | Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel | Libros Maestros | £36.94 | Travel | ⭐ 2 | 8
5- | Under the Tuscan Sun | Frances Mayes | £37.33 | Travel | ⭐ 3 | 7
6- | A Summer In Europe | Mary Elizabeth Blake | £44.34 | Travel | ⭐ 2 | 7
7- | The Great Railway Bazaar | Paul Theroux | £30.54 | Travel | ⭐ 1 | 6
8- | A Year in Provence (Provence #1) | Autor Simulado | £56.88 | Travel | ⭐ 4 | 6
9- | The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2) | Bill Bryson | £23.21 | Travel | ⭐ 1 | 3
10- | Neither Here nor There: Travels in Europe | Bill Bryson | £38.95 | Travel | ⭐ 3 | 3
11- | 1,000 Places t

DML - DATA MANIPULATION LANGUAGE

In [6]:
import sqlite3

# Funcion que guarda los datos de los libros en una base de datos SQLite.
def guardar_datos_en_db(lista_libros, libros_db="libros.db"):
    # Conectamos a la base de datos y obtenemos un cursor para ejecutar comandos SQL
    conexion = sqlite3.connect(libros_db)
    cursor = conexion.cursor()
    
    print(f"💾 Iniciando guardado de {len(lista_libros)} libros...")

    try:
        for libro in lista_libros:
            # --- A. INSERTAR CATEGORÍA ---
            # Usamos INSERT OR IGNORE para evitar duplicados
            cursor.execute("INSERT OR IGNORE INTO categorias (nombre) VALUES (?)", (libro['categoria'],))
            cursor.execute("SELECT id_categoria FROM categorias WHERE nombre = ?", (libro['categoria'],))
            cat_id = cursor.fetchone()[0] # Obtenemos el ID de la categoría recién insertada o existente para usarlo como clave foránea en la tabla de libros

            # --- B. INSERTAR AUTOR ---
            cursor.execute("INSERT OR IGNORE INTO autores (nombre_autor) VALUES (?)", (libro['autor'],))
            cursor.execute("SELECT id_autor FROM autores WHERE nombre_autor = ?", (libro['autor'],))
            autor_id = cursor.fetchone()[0] # Obtenemos el ID del autor recién insertado o existente para usarlo como clave foránea en la tabla de libros

            # --- C. INSERTAR LIBRO ---
            cursor.execute('''
                INSERT INTO libros (titulo, precio, rating, stock, link, categoria_id) 
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (libro['titulo'], libro['precio'], libro['rating'], libro['stock'], libro['link'], cat_id)) # Insertamos los datos del libro en la tabla libros
            
            libro_id = cursor.lastrowid # Obtenemos el ID del libro recién insertado para usarlo en la tabla intermedia de libro_autor

            # --- D. VINCULAR EN TABLA INTERMEDIA (Muchos a Muchos) ---
            cursor.execute('''
                INSERT OR IGNORE INTO libro_autor (libro_id, autor_id) 
                VALUES (?, ?)
            ''', (libro_id, autor_id)) # Insertamos la relación entre el libro y el autor en la tabla intermedia libro_autor

        # Confirmamos los cambios
        conexion.commit()
        print("✅ ¡Todos los datos se han guardado correctamente en la base de datos!")

    except Exception as e:
        conexion.rollback() # Si algo falla, deshacemos los cambios para no corromper la DB
        print(f"❌ Error al guardar en la DB: {e}")
    
    finally:
        conexion.close() # Cerramos la conexión a la base de datos para liberar recursos

# EJECUCIÓN: Llamamos a la función con la lista que se generó al ejecutar el scraping. Asegúrate de que la variable se llame igual (ej. lista_libros o lista_final)
guardar_datos_en_db(lista_libros)

💾 Iniciando guardado de 1000 libros...
✅ ¡Todos los datos se han guardado correctamente en la base de datos!


Conectar Base de datos!

In [7]:
import sqlite3
import time
# Conectamos a la base de datos para verificar que los datos se guardaron correctamente
libro_db = 'libros.db'
conn = sqlite3.connect(libro_db)
cursor = conn.cursor()

Consulta 1.
"Libros con 5 estrellas ordenados por precio (Desc)"

In [8]:
print("Consulta 1: Libro con 5 estrellas ordenadas por precio")

inicio = time.time() # Inicio para medir el tiempo de ejecución de la consulta

cursor.execute ("""
    SELECT titulo, precio, rating
    FROM libros
    WHERE rating = 5
    ORDER BY precio DESC;
""") # Ejecutamos la consulta SQL que nos muestra los libros con 5 estrellas ordenados por precio de forma descendente.
 
resultado = cursor.fetchall() # Variable que guarda el resultado de la consulta.

fin = time.time() # Fin para medir el tiempo de ejecución de la consulta

for row in resultado:
    print(row)

print(f"Tiempo de ejecucion: {fin - inicio:.4f} segundos") # Muestra el tiempoo que tardo la consulta.

Consulta 1: Libro con 5 estrellas ordenadas por precio
('The Barefoot Contessa Cookbook', 59.92, 5)
('Life Without a Recipe', 59.04, 5)
('Approval Junkie: Adventures in Caring Too Much', 58.81, 5)
('How to Speak Golf: An Illustrated Guide to Links Lingo', 58.32, 5)
('Digital Fortress', 58.0, 5)
('The Sound Of Love', 57.84, 5)
('Travels with Charley: In Search of America', 57.82, 5)
('El Deafo', 57.62, 5)
('H is for Hawk', 57.42, 5)
('Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine', 57.36, 5)
('The Disappearing Spoon: And Other True Tales of Madness, Love, and the History of the World from the Periodic Table of the Elements', 57.35, 5)
('Kitchens of the Great Midwest', 57.2, 5)
('A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations', 56.76, 5)
('Into the Wild', 56.7, 5)
('Eleanor & Park', 56.51, 5)
('Abstract City', 56.37, 5)
('The False Prince (The Ascendance Trilogy #1)', 56.0, 5)
('Future Shock (Future Shock #1)', 55.65, 5)
("A New Earth: Awakening to Y

CONSULTA 1 CON INDICE

In [9]:
inicio_indice = time.time() # Inicio para medir el tiempo que tarda en crear el índice
cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_rating ON libros(rating)") # Creamos un indice para optimizar las consultas.
conn.commit() # Guardamos los cambios en la base de datos para que el índice se cree correctamente
fin_indice = time.time() # Fin para medir el tiempo que tarda en crear el índice

print(f"Tiempo en crear el indice: {fin_indice - inicio_indice:.4f} segundos") # # Muestra el tiempoo que tardo la consulta.

print("Consulta 1: Libros con 5 estrellas ordenados por precio (DESC)")
inicio_consulta = time.time() # Inicio para medir el tiempo de ejecución de la consulta después de crear el índice

cursor.execute ("""
    SELECT titulo, precio, rating
    FROM libros
    WHERE rating = 5
    ORDER BY precio DESC;
""") # Ejecutamos la consulta SQL que nos muestra los libros con 5 estrellas ordenados por precio de forma descendente.

resultado = cursor.fetchall() # Variable que guarda el resultado de la consulta después de crear el índice para comparar el tiempo de ejecución con y sin índice.

fin_consulta = time.time() # Fin para medir el tiempo de ejecución de la consulta después de crear el índice

for row in resultado:
    print(row)

print(f"Tiempo de ejecucion de la consulta: {fin_consulta - inicio_consulta:.4f} segundos")

Tiempo en crear el indice: 0.0179 segundos
Consulta 1: Libros con 5 estrellas ordenados por precio (DESC)
('The Barefoot Contessa Cookbook', 59.92, 5)
('Life Without a Recipe', 59.04, 5)
('Approval Junkie: Adventures in Caring Too Much', 58.81, 5)
('How to Speak Golf: An Illustrated Guide to Links Lingo', 58.32, 5)
('Digital Fortress', 58.0, 5)
('The Sound Of Love', 57.84, 5)
('Travels with Charley: In Search of America', 57.82, 5)
('El Deafo', 57.62, 5)
('H is for Hawk', 57.42, 5)
('Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine', 57.36, 5)
('The Disappearing Spoon: And Other True Tales of Madness, Love, and the History of the World from the Periodic Table of the Elements', 57.35, 5)
('Kitchens of the Great Midwest', 57.2, 5)
('A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations', 56.76, 5)
('Into the Wild', 56.7, 5)
('Eleanor & Park', 56.51, 5)
('Abstract City', 56.37, 5)
('The False Prince (The Ascendance Trilogy #1)', 56.0, 5)
('Future Shock (Future 

Consulta 2: "Autores con 2 o mas libros."

In [10]:
print("Consulta 2: Autores con 2 o mas Libros")

cursor.execute("""
    SELECT 
        autores.nombre_autor, 
        COUNT(libro_autor.libro_id)
    FROM autores
    JOIN libro_autor ON autores.id_autor = libro_autor.autor_id
    GROUP BY autores.id_autor, autores.nombre_autor
    HAVING COUNT(libro_autor.libro_id) > 1
    ORDER BY COUNT(libro_autor.libro_id) DESC;
""") # Ejecutamos la consulta SQL para obtener los autores con 2 o mas libros.

for row in cursor.fetchall(): # Iteramos sobre el resultado de la consulta y mostramos cada fila.
    print(row)

Consulta 2: Autores con 2 o mas Libros
('Autor Simulado', 384)
('Natsuki Takaya', 8)
('Stephen King', 6)
('Sophie Kinsella', 5)
('Brian K. Vaughan', 4)
('Libros Maestros', 3)
('Arthur Conan Doyle', 3)
('Robert Galbraith', 3)
('Gene Luen Yang', 3)
('Neil Gaiman', 3)
('Brian K Vaughan', 3)
('Douglas Adams', 3)
('George R. R. Martin', 3)
('J.K. Rowling', 3)
('David Sedaris', 3)
('Bill Bryson', 2)
('John Steinbeck', 2)
('Jane Austen', 2)
('Alice Hoffman', 2)
('Philippa Gregory', 2)
('Agatha Christie', 2)
('James Patterson', 2)
('Roald Dahl', 2)
('Rob Sheffield', 2)
('Kurtis J. Wiebe', 2)
('Shannon Watters', 2)
('John Allison', 2)
('Lee Bermejo', 2)
('Kieron Gillen', 2)
('Goldy Moldavsky', 2)
('Instaread', 2)
('Anna Quindlen', 2)
('Dan Brown', 2)
('John Grisham', 2)
('Liane Moriarty', 2)
('Sarah J. Maas', 2)
('Richelle Mead', 2)
('Roshani Chokshi', 2)
('Rick Riordan', 2)
('Maggie Stiefvater', 2)
('J. K. Rowling', 2)
('Richard Dawkins', 2)
('Malcolm Gladwell', 2)
('Homer', 2)
('Brian Greene'

Consulta 3: "Libros que cuestan 50 euros para arriba."

In [11]:
print("Consulta 3: Libros que cuestan 50 euros para arriba.")

cursor.execute("""
    SELECT titulo, precio
    FROM libros
    WHERE precio > 50.0
    ORDER BY precio DESC;
""") # Ejecutamos la consulta SQL que nos muestra los libros que tienen un precio mayor a 50 euros, ordenados por precio de forma descendente para mostrar primero los libros más caros.

for row in cursor.fetchall(): # Iteramos sobre el resultado de la consulta y mostramos cada fila.
    print(row)

Consulta 3: Libros que cuestan 50 euros para arriba.
('The Perfect Play (Play by Play #1)', 59.99)
('Last One Home (New Beginnings #1)', 59.98)
('Civilization and Its Discontents', 59.95)
('The Barefoot Contessa Cookbook', 59.92)
('The Diary of a Young Girl', 59.9)
('The Bone Hunters (Lexy Vaughan & Steven Macaulay #2)', 59.71)
('Thomas Jefferson and the Tripoli Pirates: The Forgotten War That Changed American History', 59.64)
('Boar Island (Anna Pigeon #19)', 59.48)
('The Improbability of Love', 59.45)
('The Man Who Mistook His Wife for a Hat and Other Clinical Tales', 59.45)
('The Gray Rhino: How to Recognize and Act on the Obvious Dangers We Ignore', 59.15)
('Life Without a Recipe', 59.04)
('Listen to Me (Fusion #1)', 58.99)
('Unlimited Intuition Now', 58.87)
('Approval Junkie: Adventures in Caring Too Much', 58.81)
('Hamilton: The Revolution', 58.79)
('Myriad (Prentor #1)', 58.75)
('The Rose & the Dagger (The Wrath and the Dawn #2)', 58.64)
('Candide', 58.63)
('Alight (The Generati

Consulta 4: "Libros que tengan mas de 10 unidades en stock"

In [12]:
print("Consulta 4: Libros que tengan mas de 10 unidades en stock.")

cursor.execute("""
    SELECT titulo, stock
    FROM libros
    WHERE stock > 10
    ORDER BY stock DESC;
""") # Ejecutamos la consulta SQL que nos muestra los libros que tienen más de 10 unidades en stock.

for row in cursor.fetchall(): # Iteramos sobre el resultado de la consulta y mostramos cada fila.
    print(row)

Consulta 4: Libros que tengan mas de 10 unidades en stock.
('A Light in the Attic', 22)
('Tipping the Velvet', 20)
('Sharp Objects', 20)
('Soumission', 20)
('Sapiens: A Brief History of Humankind', 20)
("It's Only the Himalayas", 19)
('Chase Me (Paris Nights #2)', 19)
('Black Dust', 19)
('Birdsong: A Story in Pictures', 19)
('Rip it Up and Start Again', 19)
('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991', 19)
('How Music Works', 19)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 19)
('Mesaerion: The Best Science Fiction Stories 1800-1849', 19)
('The Requiem Red', 19)
('Set Me Free', 19)
('The Black Maria', 19)
("Shakespeare's Sonnets", 19)
('Olio', 19)
('The Dirty Little Secrets of Getting Your Dream Job', 19)
('Foolproof Preserving: A Guide to Small Batch Jams, Jellies, Pickles, Condiments, and More: A Foolproof Guide to Making Small Batch Jams, Jellies, Pickles, Condiments, and More', 19)
('In Her Wake', 19)
('Libertarianism for Begi

CONSULTA 5: "Libros con más de 3 estrellas por menos de £15, para cuando estás en bancarrota pero con estándares"

In [13]:
print("CONSULTA 5: Libros con más de 3 estrellas por menos de £15")

cursor.execute("""
    SELECT titulo, rating, precio
    FROM libros
    WHERE rating > 3 AND precio < 15.0
    ORDER BY rating DESC, precio ASC;
""") # Ejecutamos la consulta SQL que nos muestra los libros que tienen un rating mayor a 3 estrellas y un precio menor a £15.

for row in cursor.fetchall(): # Iteramos sobre el resultado de la consulta y mostramos cada fila.
    print(row)

CONSULTA 5: Libros con más de 3 estrellas por menos de £15
('An Abundance of Katherines', 5, 10.0)
('Greek Mythic History', 5, 10.23)
('The Power Greens Cookbook: 140 Delicious Superfood Recipes', 5, 11.05)
('Dear Mr. Knightley', 5, 11.21)
('The Darkest Corners', 5, 11.33)
('Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--All Under 300 Calories', 5, 11.38)
('Fruits Basket, Vol. 2 (Fruits Basket #2)', 5, 11.64)
('Old School (Diary of a Wimpy Kid #10)', 5, 11.83)
('Superman Vol. 1: Before Truth (Superman by Gene Luen Yang #1)', 5, 11.89)
('Every Heart a Doorway (Every Heart A Doorway #1)', 5, 12.16)
('The Girl You Lost', 5, 12.29)
('The Silent Wife', 5, 12.34)
('Agnostic: A Spirited Manifesto', 5, 12.51)
('The Third Wave: An Entrepreneurâ\x80\x99s Vision of the Future', 5, 12.61)
("Walt Disney's Alice in Wonderland", 5, 12.96)
('Princess Between Worlds (Wide-Awake Princess #5)', 5, 13.34)
('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)

CIERRE CONEXION BASE DE DATOS.

In [ ]:
# Cerramos la conexión a la base de datos después de realizar las consultas para liberar recursos y evitar posibles bloqueos o problemas de acceso a la base de datos en el futuro.ssss
try:
    conn.close()
    print("✅ Conexión cerrada.")
except NameError:
    print("⚠️ La variable 'conn' no existe o ya fue cerrada.")